**Note: Please "join" the competition first. Then, you can mount the dataset to the GPU. Otherwise, the notebook may encounter an error because it cannot access the dataset until you have joined the competition.**

In [1]:
# The rating for the reference answer to this question (answered by the Scientific Committee) is 0.80
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
import os
import math

In [2]:
# Load train set
file_name = '/bohr/train-d2jt/v1/data_train/training_data.dat' # This is equivalent to loading only the chemical reaction result data
# file_name = '/bohr/train-d2jt/v1/data_train' # If you want to call all the process data in the folder, you need to write your own data-reading code, and the following commit section must be modified accordingly
data = pd.read_csv(file_name)
train, val = train_test_split(data, test_size=0.05, random_state=42)  # Choose 95% of the data for training, or you can choose to cut or not cut it

In [3]:
class CustomDataset(Dataset):
    def __init__(self, initial_c, t12):
        self.input = torch.tensor(initial_c, dtype=torch.float32)
        self.label = torch.tensor(t12, dtype=torch.float32)

    def __len__(self):
        return len(self.label);

    def __getitem__(self, i):
        return self.input[i], self.label[i];

## Train phase

In [4]:
input_columns = [1, 2, 3]
initial_c = train.iloc[:, input_columns].values
output_columns = 5
t12_all = train.iloc[:, output_columns].values    

dataset = CustomDataset(initial_c, t12_all);
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)



In [18]:
class MyModel(nn.Module):
    def __init__(self):
        super(MyModel, self).__init__()
        self.fc1 = nn.Linear(3, 32)
        self.fc2 = nn.Linear(32, 64)
        self.fc3 = nn.Linear(64, 128)
        self.fc4 = nn.Linear(128, 128)
        self.fc5 = nn.Linear(128, 1)
        self.relu = nn.ReLU()
        self.elu = nn.ELU()
        self.Mish = nn.Mish()
        self.norm1 = nn.LayerNorm([32])
        self.norm2 = nn.LayerNorm([64])
        self.norm3 = nn.LayerNorm([128])
    def forward(self, x):
        x1 = self.elu(self.norm1(self.fc1(x)))
        x2 = self.Mish(self.norm2(self.fc2(x1)))
        x3 = self.relu(self.norm3(self.fc3(x2)))
        x4 = torch.nn.functional.leaky_relu(self.fc4(x3))
        x5 = self.fc5(x4)

        return x5

In [35]:
num_epochs = 2000
LR = 0.0001
model = MyModel()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

def val_model(model):
    initial_c_val = val.iloc[:, input_columns].values
    t12_all_val = val.iloc[:, output_columns].values
    model.eval()
    model = model.to(device)
    dataset_val = CustomDataset(initial_c_val, t12_all_val);
    dataloader_val = DataLoader(dataset, batch_size=32, shuffle=False)
    
    with torch.no_grad():
        tot_loss = 0.0
        batch_num = 0
        for input, label in dataloader_val:
            input = input.to(device)
            label = label.to(device)
            output = model(input)
            output = output.squeeze(1)
            loss = criterion(output, label)
            tot_loss += loss.cpu().item()
            batch_num += 1
    return tot_loss / batch_num

model.train()
model_bck = MyModel()
val_loss_min = 1e9
for epoch in range(num_epochs):
    tot_loss = 0
    batch_num = 0
    for input, label in dataloader:
        input, label = input.to(device), label.to(device)
        #print(input)
        #print(label)
        optimizer.zero_grad()
        output = model(input)
        output = torch.squeeze(output)
        loss = criterion(output, label)
        tot_loss += loss
        batch_num += 1
        loss.backward()
        optimizer.step()
    avg_loss = tot_loss / batch_num
    val_loss = val_model(model)
    if val_loss < val_loss_min:
        val_loss_min = val_loss
        model_bck.load_state_dict(model.state_dict())
    if (epoch + 1) % 10 == 0:
        print(f'epoch {epoch + 1}; train loss {avg_loss: .4f} val loss {val_loss: .4f}')

model.load_state_dict(model_bck.state_dict())
val_model(model)

## Save static model parameters

In [36]:
# Save model parameters to avoid queuing on submission
torch.save(model.state_dict(), '/personal/NOAI2025_2_model.pth') # Don't change /personal, it means it's stored in the “file” on the left.
#!cp mymodel.pth /personal   #Move the file to the folder /personal
# Instantiate a new model (structure must be the same as when saved)
#model = MyModel()  # Make sure you use the same class here that your model uses when saving the model
 
# Load parameters into the model
# model.load_state_dict(torch.load('Address_of_the_dataset_(folder)_you_created_and_model_file_name.pth'))
# model.to(device)

## Validation and test phase

### Validation set
Participants can see the A-list results on the submission platform

In [37]:
# Below is the process of validation set run
if os.environ.get('DATA_PATH'):
    data_path = os.environ.get("DATA_PATH") + "/"  
else:
    print("When the baseline is running, this error message will appear because the test set cannot be read, which is a normal phenomenon.") #When the baseline is running, this error message will appear because the test set cannot be read, which is a normal phenomenon.

data_file_name = data_path + 'data_val/val_data_question.dat'
data = pd.read_csv(data_file_name)
input_columns = [1, 2, 3]
initial_c = data.iloc[:, input_columns].values
input_val = torch.tensor(initial_c, dtype=torch.float32)
input_val = input_val.to(device)

model.eval()
tot_score = 0
pd_pred = pd.DataFrame(columns = ['Exp #', 't12_simulated'])

with torch.no_grad():
    for i in range(len(initial_c)):
        t12_pred = model(input_val[i])
        pred = t12_pred.item()
        pd_pred.loc[i, 'Exp #']= i
        pd_pred.loc[i, 't12_simulated'] = pred


pd_pred['t12_simulated'] = pd_pred['t12_simulated'].apply(lambda x: f"{x:.4e}")
pd_pred.to_csv('submission_val.csv', index=False)

### Test set
Participants are unable to access test sets or receive test set scores after submission

In [ ]:
data_file_name = data_path + 'data_test/test_data_question.dat'
data = pd.read_csv(data_file_name)
input_columns = [1, 2, 3]
initial_c = data.iloc[:, input_columns].values
input_test = torch.tensor(initial_c, dtype=torch.float32)
input_test = input_test.to(device)

model.eval()
pd_pred_test = pd.DataFrame(columns = ['Exp #', 't12_simulated'])

with torch.no_grad():
    for i in range(len(initial_c)):
        t12_pred = model(input_test[i])
        pred = t12_pred.item()
        pd_pred_test.loc[i, 'Exp #']= i
        pd_pred_test.loc[i, 't12_simulated'] = pred




pd_pred_test['t12_simulated'] = pd_pred_test['t12_simulated'].apply(lambda x: f"{x:.4e}")
pd_pred_test.to_csv('submission_test.csv', index=False)

In [ ]:
# Be sure not to delete the following code to avoid errors
import zipfile
with zipfile.ZipFile('submission.zip', 'w') as zipf:
        zipf.write('submission_val.csv')
        zipf.write('submission_test.csv')